# Building Piping Systems with the PipingBuilder DSL

In this notebook, we deep-dive into constructing piping models using Tuba's **cursor-based procedural DSL** (`PipingBuilder`). 

### What you'll learn

1. How the cursor-based approach simplifies 3D routing.
2. How to handle curves (bends) and arbitrary 3D directions.
3. How to model different structural section profiles (pipes, solid/hollow bars, cables, box beams, and catalog I-beams).
4. How to create branches (T-junctions) by referencing existing nodes.
5. How to visualize elements with their local coordinate triads.

## 1. Environment Setup & Imports

Let's import Tuba, NumPy, and PyVista, and configure the path so we can resolve the local package.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pyvista as pv

# Setup repo root for path import
REPO_ROOT = Path.cwd()
if REPO_ROOT.name.lower() == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from tuba import Model

# Enable interactive notebook rendering
# Defaults to zoomable 'client' locally; set TUBA_NOTEBOOK_BACKEND=static for nbconvert/CI.
from tuba.visualizer.notebook import configure_notebook_backend
JUPYTER_BACKEND = configure_notebook_backend()

## 2. The PipingBuilder & Simple Straight Runs

The PipingBuilder is designed as a **fluent interface** that maintains a 3D cursor (position, forward direction, and an up vector). 
You activate the builder using the context manager:
```python
with model.pipe(section=..., material=...) as b:
```
This automatically handles creating nodes, calculating segment lengths, and creating elements on the parent model.

In [ ]:
# Initialize a new model
model = Model("StraightPipe")

# Add basic carbon steel material
model.add_material("Steel", E=2.0e11, nu=0.3, alpha=1.2e-5, rho=7850)

# Add pipe section profile (DN150 / 6-inch Schedule 40)
model.add_pipe_section("DN150", OD=0.1683, WT=0.0071)

# Construct a 5-meter straight pipe
with model.pipe(section="DN150", material="Steel") as b:
    b.start([0, 0, 0]).run(5.0).end()

print("Nodes:")
for nid, node in model.nodes.items():
    print(f"  Node {nid}: {node.coords}")

print("\nElements:")
for elem in model.elements:
    print(f"  Element {elem.id}: type={elem.type}, n1={elem.n1}, n2={elem.n2}")

## 3. Bends and Direction Changes

Piping stress analysis relies heavily on elbows/bends to provide flexibility for thermal expansion. 
The PipingBuilder supports calling `.bend(radius, angle, plane)` to insert curved sections and rotate the cursor's forward direction. Supported planes are `"XY"` (horizontal) and `"XZ"` (vertical).

You can also use `.set_direction(vector)` to steer the pipe in arbitrary directions.

In [ ]:
model2 = Model("MultiLegPipe")
model2.add_material("Steel", E=2.0e11, nu=0.3, alpha=1.2e-5, rho=7850)
model2.add_pipe_section("DN150", OD=0.1683, WT=0.0071)

# Build a 3D routing leg with two 90-degree bends
with model2.pipe(section="DN150", material="Steel") as b:
    b.start([0, 0, 0])
    b.run(3.0)
    b.bend(radius=0.3, angle=90.0, plane="XY")  # turn in horizontal plane
    b.run(2.0)
    b.bend(radius=0.3, angle=90.0, plane="XZ")  # turn upwards
    b.run(1.5)
    b.end()

print("Routing node coordinates:")
for nid, node in model2.nodes.items():
    print(f"  Node {nid}: {node.coords}")

## 4. Modeling Different Section Profiles

Tuba supports coupled structural/piping models. In addition to normal pipes, you can model beams, columns, bar supports, or cables. The sections are defined as:

| Section Class | Description | Section Builder Method |
|---|---|---|
| `PipeSection` | Hollow cylinder with corrosion allowance | `run()` / `run_element(..., 'pipe_straight')` |
| `BarSection` | Solid or hollow cylinder | `bar()` |
| `CableSection` | Tension-only linear elements with pretension | `cable()` |
| `RectangularSection` | Hollow box/rectangular beams | `beam()` |
| `IBeamSection` | standard I-beams loaded from catalog | `beam()` |

Let's create a model containing all of these section types running parallel to each other.

In [ ]:
model3 = Model("MultiSectionDemo")
model3.add_material("Steel", E=2.0e11, nu=0.3, alpha=1.2e-5, rho=7850)

# Define the section profiles
model3.add_pipe_section("DN100", OD=0.1143, WT=0.006)
model3.add_bar_section("Bar50", OD=0.05, WT=0.0)  # Solid 50mm bar
model3.add_cable_section("Cable10", radius=0.01, pretension=500.0)  # 20mm cable
model3.add_rectangular_section("Box80x40", height_y=0.08, height_z=0.04, thickness_y=0.005, thickness_z=0.005)
model3.add_ibeam_section("IPE100", "IPE100")  # IPE100 standard beam from DB

# Build elements parallel to each other
# 1. Pipe straight run
with model3.pipe(section="DN100", material="Steel") as b:
    b.start([0, 0.0, 0]).run(4.0).end()

# 2. Bar straight run
with model3.pipe(section="Bar50", material="Steel") as b:
    b.start([0, 1.0, 0]).bar(4.0).end()

# 3. Cable straight run
with model3.pipe(section="Cable10", material="Steel") as b:
    b.start([0, 2.0, 0]).cable(4.0).end()

# 4. Box beam straight run
with model3.pipe(section="Box80x40", material="Steel") as b:
    b.start([0, 3.0, 0]).beam(4.0).end()

# 5. I-Beam straight run
with model3.pipe(section="IPE100", material="Steel") as b:
    b.start([0, 4.0, 0]).beam(4.0).end()

print(f"Created {len(model3.elements)} parallel elements.")

## 5. Visualizing local axes coordinate frames

Because beams and rectangular sections have asymmetrical bend/torsional stiffness, it is crucial to verify their local coordinate system orientations (triads).
We can use `plots.add_local_axes_to_plotter()` to display these frames (X-axis in Red, Y-axis in Green, Z-axis in Blue) at the midpoint of each element.

In [ ]:
from tuba.visualizer.pipeline import build_mesh_from_model, inflate_tubes
from tuba.visualizer.plots import add_local_axes_to_plotter

mesh = build_mesh_from_model(model3)
tubes = inflate_tubes(mesh, radius=0.03)

p = pv.Plotter()
p.set_background("#1a1a2e")
p.add_mesh(tubes, color="#5c6b73")

# Draw local axes at element midpoints
add_local_axes_to_plotter(p, model3, scale=0.15)
p.add_axes()
p.show(jupyter_backend=JUPYTER_BACKEND)

## 6. Branching Pipes & T-Junctions

When routing complex layouts, you often branch off from an existing pipe run. 
The PipingBuilder is designed to handle this seamlessly: if you start a new pipe run context at coordinates where a node already exists, the builder detects the node and branches off from it.

In [ ]:
model4 = Model("BranchingDemo")
model4.add_material("Steel", E=2.0e11, nu=0.3, alpha=1.2e-5, rho=7850)
model4.add_pipe_section("DN100", OD=0.1143, WT=0.006)

# 1. Create main straight run (interrupted at the midpoint [3.0, 0, 0] to create a node)
with model4.pipe(section="DN100", material="Steel") as b:
    b.start([0, 0, 0]).run(3.0).run(3.0).end()

# 2. Connect branch starting at [3.0, 0, 0] going vertical (+Z)
with model4.pipe(section="DN100", material="Steel") as b:
    b.start([3.0, 0.0, 0.0]).set_direction([0.0, 0.0, 1.0]).run(2.0).end()

print("Nodes in branching model:")
for nid, node in model4.nodes.items():
    print(f"  Node {nid}: {node.coords}")

print("\nElements in branching model:")
for elem in model4.elements:
    print(f"  Element {elem.id}: n1={elem.n1}, n2={elem.n2}")

## Key Takeaways

- The `PipingBuilder` keeps track of cursor position and directions so you don't have to manually calculate 3D trigonometry.
- Calling `.bend()` handles tangent lines and exits in the correct heading.
- General structural elements (`beam()`, `bar()`, `cable()`) can be mixed directly with normal pipes.
- Local axes display is a valuable visual validation check before running FEA solvers.